In [ ]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass, field
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl


## Pencatatan Chat dan Intent


Mencatat perilaku pemain dari chat publik. Pencatatan ini tetap tersedia untuk dipakai pada alur game.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `DEFENSIVE_INTENTS / ACCUSATION_INTENTS` | Mengelompokkan intent pembelaan dan tuduhan. | `defend` berarti membela; `offend` berarti menuduh. `neutral` tidak menambah kedua hitungan. |


In [ ]:
DEFENSIVE_INTENTS = {"defend"}
ACCUSATION_INTENTS = {"offend"}


### Riwayat Chat Satu Pemain


`PlayerChatLog` menyimpan chat dan memperbarui hitungan perilaku satu pemain.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `chats` | Menyimpan teks chat pemain. | Awalnya kosong; bertambah setiap `catat_chat` dipanggil. |
| `jumlah_menuduh / jumlah_bela_diri` | Menghitung chat berintent offend atau defend. | Awalnya 0; naik 1 pada intent yang sesuai. Ini jumlah chat, bukan jumlah pemain. |
| `giliran_diam_berturut` | Menghitung giliran tanpa chat. | `catat_diam` menambah 1; chat baru mengembalikannya ke 0. |


In [ ]:
@dataclass
class PlayerChatLog:
    """Menyimpan riwayat chat dan sinyal publik untuk satu pemain, dari sudut pandang Civilian."""

    chats: list[str] = field(default_factory=list)
    jumlah_menuduh: int = 0          # berapa kali PEMAIN INI dapat intent offend
    jumlah_bela_diri: int = 0        # berapa kali PEMAIN INI dapat intent defend
    giliran_diam_berturut: int = 0

    def catat_chat(self, teks: str, intent: str) -> None:
        """Tambahkan satu chat baru dari pemain ini dan perbarui counter berdasarkan intent-nya."""
        self.chats.append(teks)
        self.giliran_diam_berturut = 0  # baru saja chat, reset hitungan diam
        if intent in ACCUSATION_INTENTS:
            self.jumlah_menuduh += 1
        if intent in DEFENSIVE_INTENTS:
            self.jumlah_bela_diri += 1

    def catat_diam(self) -> None:
        """Panggil sekali per giliran chat ketika pemain ini tidak chat sama sekali."""
        self.giliran_diam_berturut += 1


### Riwayat Chat Banyak Pemain


`CivilianSuspicionTracker` mengelola catatan terpisah untuk setiap pemain.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `_logs / pemain` | Menghubungkan nama pemain dengan riwayatnya. | Nama baru mendapat `PlayerChatLog` sendiri, sehingga hitungan antarpemain tidak tercampur. |
| `catat_chat / catat_diam / get_log` | Memperbarui atau mengambil catatan pemain. | Pemain adalah orang yang berbicara, bukan orang yang dituduh. |


In [ ]:
class CivilianSuspicionTracker:
    """Menyimpan PlayerChatLog untuk setiap pemain yang dipantau seorang Civilian."""

    def __init__(self) -> None:
        self._logs: dict[str, PlayerChatLog] = defaultdict(PlayerChatLog)

    def catat_chat(self, pemain: str, teks: str, intent: str) -> None:
        """pemain = orang yang MENGUCAPKAN chat ini (bukan yang dituduh)."""
        self._logs[pemain].catat_chat(teks, intent)

    def catat_diam(self, pemain: str) -> None:
        self._logs[pemain].catat_diam()

    def get_log(self, pemain: str) -> PlayerChatLog:
        return self._logs[pemain]


## Sistem Fuzzy dan Penentuan Intent


### Pembentukan Aturan Fuzzy


`_build_ai_defense_fuzzy_system` membuat aturan untuk mengubah persentase penuduh menjadi skor kecenderungan intent. Fuzzy memberi derajat keanggotaan pada rentang nilai, bukan hanya pilihan ya/tidak.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `persentase_dituduh` | Input persentase pemain yang menuduh AI. | Rentang 0?100, langkah 1. Nilai 25 berarti 25% dari total pemain. |
| `fuzz.trimf(..., [a, b, c])` | Membentuk keanggotaan berbentuk segitiga. | a adalah batas kiri, b puncak keanggotaan, c batas kanan. Nilai ini menentukan bentuk rentang, bukan hasil training. |
| `nihil / minoritas / mayoritas` | Membagi input menjadi tiga kelompok. | Masing-masing menggunakan [0, 0, 1], [0, 25, 50], dan [50, 100, 100]. |
| `kecenderungan_intent` | Output skor fuzzy 0?100. | Skor rendah mengarah ke neutral, tengah ke offend, tinggi ke defend. Ini bukan probabilitas bahwa AI atau pemain berbohong. |
| `netral / offend / defend` | Membentuk kelompok skor keluaran. | Segitiganya [0, 0, 30], [20, 50, 80], dan [70, 100, 100]. |
| `rules` | Menghubungkan kelompok input dengan output. | Nihil ? netral; minoritas ? offend; mayoritas ? defend. |


In [ ]:
def _build_ai_defense_fuzzy_system() -> ctrl.ControlSystem:
    persentase_dituduh = ctrl.Antecedent(np.arange(0, 101, 1), "persentase_dituduh")
    kecenderungan_intent = ctrl.Consequent(np.arange(0, 101, 1), "kecenderungan_intent")

    # Tiga zona kasar: nyaris tidak ada yang menuduh, sebagian kecil, mayoritas.
    persentase_dituduh["nihil"] = fuzz.trimf(persentase_dituduh.universe, [0, 0, 1])
    persentase_dituduh["minoritas"] = fuzz.trimf(persentase_dituduh.universe, [0, 25, 50])
    persentase_dituduh["mayoritas"] = fuzz.trimf(persentase_dituduh.universe, [50, 100, 100])

    # Skor rendah -> neutral, tengah -> offend, tinggi -> defend.
    kecenderungan_intent["netral"] = fuzz.trimf(kecenderungan_intent.universe, [0, 0, 30])
    kecenderungan_intent["offend"] = fuzz.trimf(kecenderungan_intent.universe, [20, 50, 80])
    kecenderungan_intent["defend"] = fuzz.trimf(kecenderungan_intent.universe, [70, 100, 100])

    rules = [
        ctrl.Rule(persentase_dituduh["nihil"], kecenderungan_intent["netral"]),
        ctrl.Rule(persentase_dituduh["minoritas"], kecenderungan_intent["offend"]),
        ctrl.Rule(persentase_dituduh["mayoritas"], kecenderungan_intent["defend"]),
    ]
    return ctrl.ControlSystem(rules)


In [ ]:
# Membuat sistem aturan sekali untuk dipakai pada perhitungan skor berikutnya.
_AI_DEFENSE_FUZZY_SYSTEM = _build_ai_defense_fuzzy_system()


### Perhitungan Skor dan Label Intent


Dua fungsi berikut menghitung skor dari aturan fuzzy, kemudian mengubahnya menjadi label balasan AI.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `calculate_ai_defense_score` | Menghitung skor dari persentase tuduhan. | Input dibatasi ke 0?100 dengan `np.clip`, lalu diproses oleh simulasi fuzzy. |
| `describe_ai_defense_intent` | Mengubah skor menjadi label intent. | Skor <35 ? neutral; 35 sampai <65 ? offend; ?65 ? defend. Batas ini berlaku pada skor output, bukan langsung pada persentase penuduh. |


In [ ]:
def calculate_ai_defense_score(persentase_dituduh: float) -> float:
    """0-100: makin tinggi skor, makin condong AI harus defend (bukan offend/neutral)."""
    simulation = ctrl.ControlSystemSimulation(_AI_DEFENSE_FUZZY_SYSTEM)
    simulation.input["persentase_dituduh"] = float(np.clip(persentase_dituduh, 0, 100))
    simulation.compute()
    return float(simulation.output["kecenderungan_intent"])


In [ ]:
def describe_ai_defense_intent(score: float) -> str:
    """Ubah skor 0-100 jadi label intent AI."""
    score = float(np.clip(score, 0, 100))
    if score < 35:
        return "neutral"
    if score < 65:
        return "offend"
    return "defend"


## Pelacakan Tuduhan terhadap AI


`AISelfDefenseState` mencatat siapa yang menuduh AI dan menyediakan skor serta rekomendasi intent.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `NAMA_AI_DEFAULT / nama_ai` | Menentukan nama AI yang menjadi target tuduhan. | Default `AI`; target chat dibandingkan dengan nama ini. |
| `total_pemain` | Menjadi pembagi persentase penuduh. | 2 penuduh dari total 6 pemain menghasilkan 33,33%. Dengan jumlah penuduh tetap, total pemain lebih besar menghasilkan persentase lebih kecil. |
| `pemain_penuduh_ai` | Menyimpan pemain unik yang menuduh AI. | Berbentuk set: tuduhan berulang oleh pemain yang sama tidak menambah jumlah pemain penuduh. |
| `jumlah_menuduh_global` | Menghitung tuduhan ke pemain selain AI. | Naik 1 per tuduhan; menjadi konteks tambahan dan tidak masuk rumus skor defense. |
| `catat_chat` | Memeriksa intent dan target tuduhan. | Hanya memproses offend dengan target yang diberikan. |
| `persentase_dituduh` | Menghitung proporsi pemain penuduh. | Jumlah penuduh unik / total pemain ? 100. Jika total pemain ?0, hasilnya 0. |
| `skor_defense / rekomendasi_intent_ai` | Mengambil hasil dari fungsi fuzzy di cell atas. | Mengembalikan skor 0?100 atau label neutral/offend/defend. |


In [ ]:
NAMA_AI_DEFAULT = "AI"


In [ ]:
@dataclass
class AISelfDefenseState:
    """Melacak tuduhan yang mengarah ke AI, untuk menentukan intent balasan AI.

    - persentase_dituduh: persentase pemain (dari total pemain) yang sudah
      pernah menuduh AI secara langsung.
    - jumlah_menuduh_global: berapa kali PEMAIN LAIN menuduh PEMAIN LAIN
      (bukan menuduh AI) -- dipakai sebagai sinyal tambahan/konteks, bukan
      untuk menentukan intent AI sendiri.
    """

    total_pemain: int
    nama_ai: str = NAMA_AI_DEFAULT
    pemain_penuduh_ai: set[str] = field(default_factory=set)
    jumlah_menuduh_global: int = 0

    def catat_chat(self, pemain: str, intent: str, target: str | None = None) -> None:
        """pemain = siapa yang bicara. target = siapa yang dituduh (hanya relevan kalau intent == 'offend').

        Kalau target == nama AI, dicatat sebagai tuduhan terhadap AI.
        Kalau target pemain lain (bukan AI), dihitung ke jumlah_menuduh_global.
        """
        if intent != "offend" or target is None:
            return
        if target == self.nama_ai:
            self.pemain_penuduh_ai.add(pemain)
        else:
            self.jumlah_menuduh_global += 1

    @property
    def persentase_dituduh(self) -> float:
        """Persentase pemain (dari total pemain) yang sudah menuduh AI."""
        if self.total_pemain <= 0:
            return 0.0
        return len(self.pemain_penuduh_ai) / self.total_pemain * 100

    def rekomendasi_intent_ai(self) -> str:
        """Label intent AI (neutral/offend/defend) berdasarkan fuzzy score."""
        score = calculate_ai_defense_score(self.persentase_dituduh)
        return describe_ai_defense_intent(score)

    def skor_defense(self) -> float:
        """Skor mentah 0-100, kalau butuh nilai gradasi (bukan cuma label)."""
        return calculate_ai_defense_score(self.persentase_dituduh)


## Simulasi dan Hasil


Contoh ini menjalankan kode di atas tanpa membaca modul lokal terpisah.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `state` | Menyimpan keadaan contoh game. | Total 6 pemain; Raka dan Sinta menuduh AI, sedangkan Budi menuduh Vino. |
| `skor / label` | Menampilkan skor dan rekomendasi balasan. | Dua penuduh AI menghasilkan 33,33%; tuduhan Budi hanya menambah hitungan global. |


In [ ]:
print("Fuzzy HOSTAGE siap: hanya memakai sinyal publik dan berjalan di CPU.")

# Simulasi: 6 pemain total, sebagian menuduh AI, sebagian menuduh pemain lain.
state = AISelfDefenseState(total_pemain=6, nama_ai="AI")

state.catat_chat("Raka", "offend", target="AI")
state.catat_chat("Sinta", "offend", target="AI")
state.catat_chat("Budi", "offend", target="Vino")   # nuduh pemain lain, bukan AI

print(f"Persentase pemain yang menuduh AI: {state.persentase_dituduh:.2f}%")
print(f"Jumlah tuduhan ke sesama pemain (bukan ke AI): {state.jumlah_menuduh_global}")

skor = state.skor_defense()
label = state.rekomendasi_intent_ai()
print(f"Skor kecenderungan intent AI: {skor:.2f}% -> intent yang direkomendasikan: {label.upper()}")

Fuzzy HOSTAGE siap: hanya memakai sinyal publik dan berjalan di CPU.
Persentase pemain yang menuduh AI: 33.33%
Jumlah tuduhan ke sesama pemain (bukan ke AI): 1
Skor kecenderungan intent AI: 50.00% -> intent yang direkomendasikan: OFFEND
